# Aave v3 Long-Short Strategy: Liquidation Risk Analysis via First-Hitting Times

This notebook demonstrates the analysis of liquidation risk for a **long stablecoin / short volatile asset** strategy on Aave v3 using spectrally negative Lévy processes and first-hitting time theory.

## Strategy Overview

**Long Stablecoin / Short Volatile Asset** on Aave v3:
- **Collateral**: Stablecoins (USDC, DAI) - minimal price volatility
- **Debt**: Volatile assets (ETH, WBTC) - exposure to downside risk

This strategy profits when volatile assets decline, as debt value decreases while collateral remains stable.

### Risk Profile
- **Favorable scenario**: Volatile asset price drops → debt value decreases → Health Factor increases
- **Adverse scenario**: Volatile asset price rises → debt value increases → Health Factor decreases → **liquidation**

When shorting volatile assets, the health factor *decreases* when asset prices rise (opposite of the typical long-volatile-collateral scenario).

## Mathematical Framework Preview

The **Health Factor** on Aave is defined as:

$$H = \frac{\sum_i (\text{Collateral}_i \times \text{Price}_i \times \text{LT}_i)}{\sum_j (\text{Debt}_j \times \text{Price}_j)}$$

where $\text{LT}_i$ is the liquidation threshold for asset $i$.

We model the **log-health factor** $X_t = \log(H_t)$ as a spectrally negative Lévy process:

$$X_t = X_0 + \mu t + \sigma W_t - \sum_{i=1}^{N_t} Y_i$$

where:
- $\mu$: drift coefficient
- $\sigma$: diffusion coefficient
- $W_t$: standard Brownian motion
- $N_t \sim \text{Poisson}(\lambda t)$: jump count process
- $Y_i$: shifted exponential jump sizes

**Liquidation occurs** when $X_t \leq 0$ (equivalently, $H_t \leq 1$).

In [ ]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')

# Scientific computing
import numpy as np
import pandas as pd
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# crypto-fht imports
from crypto_fht.core.levy_process import (
    LevyParameters,
    laplace_exponent,
    laplace_exponent_derivative,
    simulate_path,
)
from crypto_fht.core.wiener_hopf import WienerHopfFactorization
from crypto_fht.core.scale_function import ScaleFunction
from crypto_fht.core.first_hitting_time import FirstHittingTime
from crypto_fht.calibration.mle_estimator import LevyMLEEstimator
from crypto_fht.visualization.matplotlib_plots import (
    plot_levy_paths,
    plot_first_hitting_time_distribution,
    plot_liquidation_heatmap,
)

print("All imports successful!")

In [ ]:
# Configuration
TIME_HORIZON_DAYS = 90
DT_DAYS = 1 / 365  # Daily observations
N_SIMULATION_DAYS = 730  # 2 years of data for calibration
RANDOM_SEED = 42

# Set random seed for reproducibility
RNG = np.random.default_rng(RANDOM_SEED)

print(f"Configuration:")
print(f"  Time horizon: {TIME_HORIZON_DAYS} days")
print(f"  Simulation data: {N_SIMULATION_DAYS} days ({N_SIMULATION_DAYS/365:.1f} years)")
print(f"  Random seed: {RANDOM_SEED}")

---
# Section 2: Mock Aave v3 Data and Position Setup

## Aave v3 Risk Parameters

Each asset on Aave has specific risk parameters:

| Asset | LTV | Liquidation Threshold | Liquidation Bonus |
|-------|-----|----------------------|-------------------|
| USDC  | 77% | 80%                  | 4.5%              |
| DAI   | 67% | 77%                  | 4%                |
| ETH   | 80% | 82.5%                | 5%                |
| WBTC  | 70% | 75%                  | 6.25%             |

- **LTV (Loan-to-Value)**: Maximum borrowing power as a fraction of collateral value
- **Liquidation Threshold**: Health factor threshold below which liquidation can occur
- **Liquidation Bonus**: Incentive for liquidators (paid from collateral)

In [ ]:
# Mock Aave v3 reserve parameters
AAVE_RESERVES = {
    'USDC': {
        'ltv': 0.77,
        'liquidation_threshold': 0.80,
        'liquidation_bonus': 0.045,
        'price_usd': 1.00,
        'is_stablecoin': True,
    },
    'DAI': {
        'ltv': 0.67,
        'liquidation_threshold': 0.77,
        'liquidation_bonus': 0.04,
        'price_usd': 1.00,
        'is_stablecoin': True,
    },
    'ETH': {
        'ltv': 0.80,
        'liquidation_threshold': 0.825,
        'liquidation_bonus': 0.05,
        'price_usd': 2500.0,  # Example price
        'is_stablecoin': False,
    },
    'WBTC': {
        'ltv': 0.70,
        'liquidation_threshold': 0.75,
        'liquidation_bonus': 0.0625,
        'price_usd': 45000.0,  # Example price
        'is_stablecoin': False,
    },
}

# Display as DataFrame
reserves_df = pd.DataFrame(AAVE_RESERVES).T
reserves_df

In [ ]:
# Define two position pairs

def compute_health_factor(collateral_value, collateral_lt, debt_value):
    """Compute Aave health factor."""
    if debt_value == 0:
        return float('inf')
    return (collateral_value * collateral_lt) / debt_value

# Pair 1: Long USDC / Short ETH
PAIR_1 = {
    'name': 'USDC/ETH',
    'collateral': 'USDC',
    'collateral_amount': 100_000,  # $100k USDC
    'debt': 'ETH',
    'debt_amount_usd': 60_000,  # $60k worth of ETH
}

# Pair 2: Long DAI / Short WBTC
PAIR_2 = {
    'name': 'DAI/WBTC',
    'collateral': 'DAI',
    'collateral_amount': 100_000,  # $100k DAI
    'debt': 'WBTC',
    'debt_amount_usd': 60_000,  # $60k worth of WBTC
}

# Compute initial health factors
for pair in [PAIR_1, PAIR_2]:
    collateral_lt = AAVE_RESERVES[pair['collateral']]['liquidation_threshold']
    collateral_value = pair['collateral_amount'] * AAVE_RESERVES[pair['collateral']]['price_usd']
    debt_value = pair['debt_amount_usd']
    
    hf = compute_health_factor(collateral_value, collateral_lt, debt_value)
    pair['initial_hf'] = hf
    pair['log_hf'] = np.log(hf)
    
    print(f"{pair['name']}:")
    print(f"  Collateral: ${collateral_value:,.0f} {pair['collateral']} (LT={collateral_lt:.1%})")
    print(f"  Debt: ${debt_value:,.0f} {pair['debt']}")
    print(f"  Health Factor: {hf:.4f}")
    print(f"  Log Health Factor: {pair['log_hf']:.4f}")
    print()

### Understanding the Long-Short Strategy

For a **long stablecoin / short volatile** position:

$$H = \frac{\text{USDC} \times 1.0 \times \text{LT}_{\text{USDC}}}{\text{ETH\_debt} \times P_{\text{ETH}}}$$

When the volatile asset price $P_{\text{ETH}}$ **increases**:
- Debt value increases (denominator grows)
- Health factor **decreases**
- Risk of liquidation **increases**

This is the **opposite** of a typical long-volatile-collateral position!

In [ ]:
# Summary table
summary_data = {
    'Pair': [PAIR_1['name'], PAIR_2['name']],
    'Collateral': [f"${PAIR_1['collateral_amount']:,} {PAIR_1['collateral']}",
                   f"${PAIR_2['collateral_amount']:,} {PAIR_2['collateral']}"],
    'Debt': [f"${PAIR_1['debt_amount_usd']:,} {PAIR_1['debt']}",
             f"${PAIR_2['debt_amount_usd']:,} {PAIR_2['debt']}"],
    'LT': [AAVE_RESERVES[PAIR_1['collateral']]['liquidation_threshold'],
           AAVE_RESERVES[PAIR_2['collateral']]['liquidation_threshold']],
    'Initial HF': [PAIR_1['initial_hf'], PAIR_2['initial_hf']],
    'Log HF': [PAIR_1['log_hf'], PAIR_2['log_hf']],
}

summary_df = pd.DataFrame(summary_data)
summary_df.set_index('Pair', inplace=True)
summary_df

---
# Section 3: Simulating Crypto Returns with Jump-Diffusion

## Why Jump-Diffusion?

Cryptocurrency returns exhibit several characteristics that standard Gaussian models fail to capture:

1. **Heavy tails**: Extreme moves are far more frequent than normal distributions predict
2. **Negative skewness**: Crashes tend to be larger than rallies
3. **Sudden large moves**: Flash crashes, liquidation cascades, protocol exploits
4. **Volatility clustering**: Periods of high and low volatility

The **spectrally negative Lévy process** captures these features through:
- Continuous diffusion component (Brownian motion)
- Downward jumps (compound Poisson process)
- Shifted exponential jump sizes (minimum jump floor)

In [ ]:
def generate_crypto_returns(
    n_days: int,
    mu: float,
    sigma: float,
    lambda_: float,
    eta: float,
    delta: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """Generate synthetic crypto returns with jump-diffusion dynamics.
    
    Parameters
    ----------
    n_days : int
        Number of daily observations
    mu : float
        Drift coefficient (annualized)
    sigma : float
        Volatility (annualized)
    lambda_ : float
        Jump intensity (jumps per year)
    eta : float
        Exponential rate for jump size distribution
    delta : float
        Minimum jump size (shift parameter)
    rng : np.random.Generator
        Random number generator
        
    Returns
    -------
    np.ndarray
        Array of daily log-returns
    """
    dt = 1 / 365  # Daily time step
    returns = np.zeros(n_days)
    
    for i in range(n_days):
        # Diffusion component
        dW = rng.standard_normal() * np.sqrt(dt)
        diffusion = mu * dt + sigma * dW
        
        # Jump component (Poisson arrivals)
        n_jumps = rng.poisson(lambda_ * dt)
        jump_loss = 0.0
        if n_jumps > 0:
            # Shifted exponential: Y = delta + Exp(eta)
            jump_sizes = delta + rng.exponential(1.0 / eta, size=n_jumps)
            jump_loss = np.sum(jump_sizes)
        
        # Combined return (jumps are negative = losses)
        returns[i] = diffusion - jump_loss
    
    return returns

print("Return generator defined.")

In [ ]:
# True parameters for simulation (these represent the "market reality")
# We'll calibrate from simulated data and compare to these
# Parameters are chosen to have slightly positive effective drift (realistic for risk analysis)

TRUE_PARAMS_ETH = {
    'mu': 0.40,       # 40% annual drift (typical crypto bull market)
    'sigma': 0.50,    # 50% annualized volatility
    'lambda_': 3.0,   # 3 jumps per year (~1 per quarter)
    'eta': 30.0,      # Mean exponential component = 1/30 ≈ 0.033
    'delta': 0.01,    # 1% minimum jump size
}
# E[Y] = 0.01 + 0.033 = 0.043
# Effective drift ETH: 0.40 - 3*0.043 = 0.40 - 0.13 = +0.27

TRUE_PARAMS_WBTC = {
    'mu': 0.30,       # 30% annual drift
    'sigma': 0.45,    # 45% annualized volatility
    'lambda_': 2.5,   # 2.5 jumps per year
    'eta': 25.0,      # Mean exponential component = 1/25 = 0.04
    'delta': 0.015,   # 1.5% minimum jump size
}
# E[Y] = 0.015 + 0.04 = 0.055
# Effective drift WBTC: 0.30 - 2.5*0.055 = 0.30 - 0.1375 = +0.16

# Generate 2 years of daily returns for each asset
returns_eth = generate_crypto_returns(
    N_SIMULATION_DAYS,
    **TRUE_PARAMS_ETH,
    rng=RNG,
)

returns_wbtc = generate_crypto_returns(
    N_SIMULATION_DAYS,
    **TRUE_PARAMS_WBTC,
    rng=RNG,
)

print(f"Generated {len(returns_eth)} daily returns for ETH")
print(f"Generated {len(returns_wbtc)} daily returns for WBTC")

# Show effective drifts
eff_eth = TRUE_PARAMS_ETH['mu'] - TRUE_PARAMS_ETH['lambda_'] * (TRUE_PARAMS_ETH['delta'] + 1/TRUE_PARAMS_ETH['eta'])
eff_wbtc = TRUE_PARAMS_WBTC['mu'] - TRUE_PARAMS_WBTC['lambda_'] * (TRUE_PARAMS_WBTC['delta'] + 1/TRUE_PARAMS_WBTC['eta'])
print(f"\nETH effective drift: {eff_eth:.4f} (positive → survival possible)")
print(f"WBTC effective drift: {eff_wbtc:.4f} (positive → survival possible)")

In [ ]:
# Visualize return distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ETH histogram
ax1 = axes[0, 0]
ax1.hist(returns_eth, bins=50, density=True, alpha=0.7, color='blue', label='ETH')
x = np.linspace(returns_eth.min(), returns_eth.max(), 100)
normal_pdf = stats.norm.pdf(x, np.mean(returns_eth), np.std(returns_eth))
ax1.plot(x, normal_pdf, 'r--', linewidth=2, label='Normal fit')
ax1.set_xlabel('Daily Return')
ax1.set_ylabel('Density')
ax1.set_title('ETH Return Distribution')
ax1.legend()

# WBTC histogram
ax2 = axes[0, 1]
ax2.hist(returns_wbtc, bins=50, density=True, alpha=0.7, color='orange', label='WBTC')
x = np.linspace(returns_wbtc.min(), returns_wbtc.max(), 100)
normal_pdf = stats.norm.pdf(x, np.mean(returns_wbtc), np.std(returns_wbtc))
ax2.plot(x, normal_pdf, 'r--', linewidth=2, label='Normal fit')
ax2.set_xlabel('Daily Return')
ax2.set_ylabel('Density')
ax2.set_title('WBTC Return Distribution')
ax2.legend()

# ETH Q-Q plot
ax3 = axes[1, 0]
stats.probplot(returns_eth, dist="norm", plot=ax3)
ax3.set_title('ETH Q-Q Plot (vs Normal)')

# WBTC Q-Q plot
ax4 = axes[1, 1]
stats.probplot(returns_wbtc, dist="norm", plot=ax4)
ax4.set_title('WBTC Q-Q Plot (vs Normal)')

plt.tight_layout()
plt.show()

print("\nNote: Q-Q plots show deviation from normality in the tails (heavy tails from jumps).")

In [ ]:
# Compute return statistics
def compute_return_stats(returns, name):
    """Compute and display return statistics."""
    annualized_vol = np.std(returns) * np.sqrt(365)
    return {
        'Asset': name,
        'Mean (daily)': f"{np.mean(returns):.6f}",
        'Std (daily)': f"{np.std(returns):.6f}",
        'Vol (ann.)': f"{annualized_vol:.2%}",
        'Skewness': f"{stats.skew(returns):.3f}",
        'Kurtosis': f"{stats.kurtosis(returns):.3f}",
        'Min': f"{np.min(returns):.4f}",
        'Max': f"{np.max(returns):.4f}",
    }

stats_eth = compute_return_stats(returns_eth, 'ETH')
stats_wbtc = compute_return_stats(returns_wbtc, 'WBTC')

stats_df = pd.DataFrame([stats_eth, stats_wbtc])
stats_df.set_index('Asset', inplace=True)
stats_df

### Key Observations

1. **Negative skewness**: Both assets show left skew (more extreme negative returns)
2. **Excess kurtosis**: Values significantly above 0 indicate heavy tails
3. **Q-Q deviation**: Clear departure from normality in the tails

These characteristics justify using a Lévy process model rather than simple Gaussian assumptions.

In [ ]:
# Time series plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Cumulative returns (price paths)
cum_eth = np.cumsum(returns_eth)
cum_wbtc = np.cumsum(returns_wbtc)

days = np.arange(N_SIMULATION_DAYS)

ax1 = axes[0]
ax1.plot(days, cum_eth, 'b-', alpha=0.8, label='ETH')
ax1.plot(days, cum_wbtc, 'orange', alpha=0.8, label='WBTC')
ax1.set_ylabel('Cumulative Log Return')
ax1.set_title('Simulated Price Paths (Log Scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Daily returns
ax2 = axes[1]
ax2.bar(days, returns_eth, alpha=0.6, width=1, color='blue', label='ETH')
ax2.bar(days, returns_wbtc, alpha=0.4, width=1, color='orange', label='WBTC')
ax2.set_xlabel('Day')
ax2.set_ylabel('Daily Return')
ax2.set_title('Daily Returns (showing jump events)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Section 4: Mathematical Background - Lévy Processes

## Spectrally Negative Lévy Process

A **spectrally negative Lévy process** is a stochastic process with:
- Stationary, independent increments
- Right-continuous paths with left limits (càdlàg)
- **Only downward jumps** (no positive jumps)

We model the log-health factor as:

$$X_t = X_0 + \mu t + \sigma W_t - \sum_{i=1}^{N_t} Y_i$$

where:
- $\mu \in \mathbb{R}$: drift coefficient
- $\sigma \geq 0$: diffusion coefficient
- $W_t$: standard Brownian motion
- $N_t \sim \text{Poisson}(\lambda t)$: jump count process with intensity $\lambda > 0$
- $Y_i \geq 0$: independent jump sizes

## Shifted Exponential Jump Distribution

The jump sizes follow a **shifted exponential distribution**:

$$Y = \delta + Z, \quad Z \sim \text{Exp}(\eta)$$

where:
- $\delta \geq 0$: minimum jump size (shift parameter)
- $\eta > 0$: exponential rate parameter

The probability density function is:

$$f_Y(y) = \eta e^{-\eta(y-\delta)} \mathbf{1}_{y \geq \delta}$$

**Moments:**
$$\mathbb{E}[Y] = \delta + \frac{1}{\eta}$$
$$\text{Var}(Y) = \frac{1}{\eta^2}$$

The shift parameter $\delta$ ensures a minimum jump size, capturing the "floor" of market dislocations (e.g., no small jumps - when markets gap, they gap significantly).

## Laplace Exponent

The **Laplace exponent** $\psi(\theta)$ encodes all distributional information about the process:

$$\mathbb{E}[e^{\theta X_t}] = e^{t \psi(\theta)}$$

For our spectrally negative Lévy process with shifted exponential jumps:

$$\boxed{\psi(\theta) = \mu\theta + \frac{\sigma^2}{2}\theta^2 + \lambda\left(e^{-\theta\delta} \cdot \frac{\eta}{\eta+\theta} - 1\right)}$$

**Components:**
1. $\mu\theta$: linear drift
2. $\frac{\sigma^2}{2}\theta^2$: quadratic term from Brownian motion
3. $\lambda(e^{-\theta\delta} \cdot \frac{\eta}{\eta+\theta} - 1)$: jump contribution

The factor $e^{-\theta\delta}$ accounts for the guaranteed minimum jump of size $\delta$.

In [ ]:
# Create LevyParameters objects with true parameters
params_eth_true = LevyParameters(
    mu=TRUE_PARAMS_ETH['mu'],
    sigma=TRUE_PARAMS_ETH['sigma'],
    lambda_=TRUE_PARAMS_ETH['lambda_'],
    eta=TRUE_PARAMS_ETH['eta'],
    delta=TRUE_PARAMS_ETH['delta'],
)

params_wbtc_true = LevyParameters(
    mu=TRUE_PARAMS_WBTC['mu'],
    sigma=TRUE_PARAMS_WBTC['sigma'],
    lambda_=TRUE_PARAMS_WBTC['lambda_'],
    eta=TRUE_PARAMS_WBTC['eta'],
    delta=TRUE_PARAMS_WBTC['delta'],
)

print("ETH True Parameters:")
print(f"  Mean jump size: E[Y] = δ + 1/η = {params_eth_true.mean_jump_size:.4f}")
print(f"  Effective drift: μ - λE[Y] = {params_eth_true.effective_drift:.4f}")
print()
print("WBTC True Parameters:")
print(f"  Mean jump size: E[Y] = δ + 1/η = {params_wbtc_true.mean_jump_size:.4f}")
print(f"  Effective drift: μ - λE[Y] = {params_wbtc_true.effective_drift:.4f}")

In [ ]:
# Plot Laplace exponent
fig, ax = plt.subplots(figsize=(10, 6))

# Theta range (avoiding singularity at theta = -eta)
theta = np.linspace(-params_eth_true.eta + 0.5, 10, 200)

psi_eth = np.array([float(np.real(laplace_exponent(t, params_eth_true))) for t in theta])
psi_wbtc = np.array([float(np.real(laplace_exponent(t, params_wbtc_true))) for t in theta])

ax.plot(theta, psi_eth, 'b-', linewidth=2, label='ETH ψ(θ)')
ax.plot(theta, psi_wbtc, 'orange', linewidth=2, label='WBTC ψ(θ)')

# Mark key features
ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)

# Mark Φ(q) for q = 1, 5
wh_eth = WienerHopfFactorization(params_eth_true)
for q in [1.0, 5.0]:
    phi_q = wh_eth.phi(q)
    ax.scatter([phi_q], [q], color='red', s=100, zorder=5)
    ax.annotate(f'Φ({q})={phi_q:.2f}', (phi_q, q), textcoords="offset points", 
                xytext=(10, 5), fontsize=10)

ax.set_xlabel('θ', fontsize=12)
ax.set_ylabel('ψ(θ)', fontsize=12)
ax.set_title('Laplace Exponent ψ(θ) for Shifted Exponential Lévy Process', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 50)

plt.tight_layout()
plt.show()

print("Red dots: Φ(q) values where ψ(Φ(q)) = q (roots of the Laplace exponent shifted by q)")

## Effective Drift

The **effective drift** determines the long-term behavior of the process:

$$\mu_{\text{eff}} = \mu - \lambda \cdot \mathbb{E}[Y] = \mu - \lambda\left(\delta + \frac{1}{\eta}\right)$$

**Interpretation:**
- If $\mu_{\text{eff}} < 0$: Process drifts downward on average → **eventual liquidation almost sure**
- If $\mu_{\text{eff}} > 0$: Process may survive indefinitely
- If $\mu_{\text{eff}} = 0$: Null recurrent case

For our crypto parameters, both processes have negative effective drift, meaning liquidation is a matter of *when*, not *if* (without intervention).

In [ ]:
# Parameter sensitivity visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

base_params = dict(mu=0.40, sigma=0.50, lambda_=3.0, eta=30.0, delta=0.01)

def plot_density_for_param(ax, param_name, param_values, label_format):
    """Plot density for different parameter values."""
    colors = plt.cm.viridis(np.linspace(0, 1, len(param_values)))
    
    for val, color in zip(param_values, colors):
        # Create modified parameters
        params = base_params.copy()
        params[param_name] = val
        
        # Generate returns
        rets = generate_crypto_returns(365, **params, rng=np.random.default_rng(42))
        
        ax.hist(rets, bins=30, density=True, alpha=0.3, color=color,
                label=label_format.format(val))
    
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# Vary each parameter
plot_density_for_param(axes[0, 0], 'mu', [0.0, 0.40, 0.80], 'μ={:.2f}')
axes[0, 0].set_title('Effect of Drift (μ)')

plot_density_for_param(axes[0, 1], 'sigma', [0.25, 0.50, 0.75], 'σ={:.2f}')
axes[0, 1].set_title('Effect of Volatility (σ)')

plot_density_for_param(axes[0, 2], 'lambda_', [1.0, 3.0, 6.0], 'λ={:.0f}')
axes[0, 2].set_title('Effect of Jump Intensity (λ)')

plot_density_for_param(axes[1, 0], 'eta', [15.0, 30.0, 60.0], 'η={:.0f}')
axes[1, 0].set_title('Effect of Jump Rate (η)')

plot_density_for_param(axes[1, 1], 'delta', [0.005, 0.01, 0.02], 'δ={:.3f}')
axes[1, 1].set_title('Effect of Min Jump Size (δ)')

# Hide last subplot
axes[1, 2].axis('off')
axes[1, 2].text(0.5, 0.5, 'Base Parameters:\n\n' + 
                '\n'.join([f"{k}={v}" for k, v in base_params.items()]),
                ha='center', va='center', fontsize=12, family='monospace')

plt.suptitle('Parameter Sensitivity Analysis', fontsize=14)
plt.tight_layout()
plt.show()

---
# Section 5: Calibration via Maximum Likelihood Estimation

## EM Algorithm for Jump-Diffusion

The **Expectation-Maximization (EM) algorithm** treats jump occurrences as latent variables:

### E-step
Compute posterior probability that observation $i$ contains a jump:

$$P(\text{jump at } t_i | r_i, \theta) \propto \frac{p_{\text{jump}} \cdot f_{\text{jump}}(r_i)}{p_{\text{jump}} \cdot f_{\text{jump}}(r_i) + (1-p_{\text{jump}}) \cdot f_{\text{diff}}(r_i)}$$

### M-step
Update parameters given expected jump indicators:
- Update $\mu, \sigma$ from non-jump observations
- Update $\lambda$ from expected number of jumps
- Update $\eta, \delta$ from expected jump sizes

The algorithm iterates until convergence (log-likelihood stabilizes).

In [ ]:
# Calibrate ETH parameters from simulated returns
estimator_eth = LevyMLEEstimator(dt=1/365, max_iter=100, tol=1e-6)

print("Calibrating ETH parameters...")
result_eth = estimator_eth.fit(returns_eth)

print("\n" + "="*60)
print("ETH CALIBRATION RESULTS")
print("="*60)
print(f"\nConverged: {result_eth.converged}")
print(f"Iterations: {result_eth.n_iterations}")
print(f"\nEstimated Parameters:")
print(f"  μ (drift):      {result_eth.params.mu:.6f}  (true: {TRUE_PARAMS_ETH['mu']:.6f})")
print(f"  σ (volatility): {result_eth.params.sigma:.6f}  (true: {TRUE_PARAMS_ETH['sigma']:.6f})")
print(f"  λ (intensity):  {result_eth.params.lambda_:.6f}  (true: {TRUE_PARAMS_ETH['lambda_']:.6f})")
print(f"  η (exp rate):   {result_eth.params.eta:.6f}  (true: {TRUE_PARAMS_ETH['eta']:.6f})")
print(f"  δ (min jump):   {result_eth.params.delta:.6f}  (true: {TRUE_PARAMS_ETH['delta']:.6f})")
print(f"\nModel Fit:")
print(f"  Log-likelihood: {result_eth.log_likelihood:.2f}")
print(f"  AIC: {result_eth.aic:.2f}")
print(f"  BIC: {result_eth.bic:.2f}")

In [ ]:
# Calibrate WBTC parameters
estimator_wbtc = LevyMLEEstimator(dt=1/365, max_iter=100, tol=1e-6)

print("Calibrating WBTC parameters...")
result_wbtc = estimator_wbtc.fit(returns_wbtc)

print("\n" + "="*60)
print("WBTC CALIBRATION RESULTS")
print("="*60)
print(f"\nConverged: {result_wbtc.converged}")
print(f"Iterations: {result_wbtc.n_iterations}")
print(f"\nEstimated Parameters:")
print(f"  μ (drift):      {result_wbtc.params.mu:.6f}  (true: {TRUE_PARAMS_WBTC['mu']:.6f})")
print(f"  σ (volatility): {result_wbtc.params.sigma:.6f}  (true: {TRUE_PARAMS_WBTC['sigma']:.6f})")
print(f"  λ (intensity):  {result_wbtc.params.lambda_:.6f}  (true: {TRUE_PARAMS_WBTC['lambda_']:.6f})")
print(f"  η (exp rate):   {result_wbtc.params.eta:.6f}  (true: {TRUE_PARAMS_WBTC['eta']:.6f})")
print(f"  δ (min jump):   {result_wbtc.params.delta:.6f}  (true: {TRUE_PARAMS_WBTC['delta']:.6f})")
print(f"\nModel Fit:")
print(f"  Log-likelihood: {result_wbtc.log_likelihood:.2f}")
print(f"  AIC: {result_wbtc.aic:.2f}")
print(f"  BIC: {result_wbtc.bic:.2f}")

In [ ]:
# Create comparison table
comparison_data = {
    'Parameter': ['μ', 'σ', 'λ', 'η', 'δ'],
    'ETH True': [TRUE_PARAMS_ETH['mu'], TRUE_PARAMS_ETH['sigma'], TRUE_PARAMS_ETH['lambda_'],
                 TRUE_PARAMS_ETH['eta'], TRUE_PARAMS_ETH['delta']],
    'ETH Est.': [result_eth.params.mu, result_eth.params.sigma, result_eth.params.lambda_,
                 result_eth.params.eta, result_eth.params.delta],
    'WBTC True': [TRUE_PARAMS_WBTC['mu'], TRUE_PARAMS_WBTC['sigma'], TRUE_PARAMS_WBTC['lambda_'],
                  TRUE_PARAMS_WBTC['eta'], TRUE_PARAMS_WBTC['delta']],
    'WBTC Est.': [result_wbtc.params.mu, result_wbtc.params.sigma, result_wbtc.params.lambda_,
                  result_wbtc.params.eta, result_wbtc.params.delta],
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df.set_index('Parameter', inplace=True)
comparison_df

In [ ]:
# Calibration diagnostics - ETH
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Histogram with fitted density
ax1 = axes[0, 0]
ax1.hist(returns_eth, bins=50, density=True, alpha=0.7, label='Observed')
x = np.linspace(np.min(returns_eth), np.max(returns_eth), 100)
# Approximate with normal for visualization
theoretical = stats.norm.pdf(x, loc=result_eth.params.mu / 365, scale=result_eth.params.sigma / np.sqrt(365))
ax1.plot(x, theoretical, 'r-', linewidth=2, label='Fitted (diffusion only)')
ax1.set_xlabel('Return')
ax1.set_ylabel('Density')
ax1.set_title('ETH Return Distribution')
ax1.legend()

# Q-Q plot
ax2 = axes[0, 1]
stats.probplot(returns_eth, dist="norm", plot=ax2)
ax2.set_title('ETH Q-Q Plot')

# Autocorrelation
ax3 = axes[1, 0]
n_lags = min(30, len(returns_eth) // 4)
acf = np.correlate(returns_eth - np.mean(returns_eth), returns_eth - np.mean(returns_eth), mode="full")
acf = acf[len(returns_eth)-1:len(returns_eth)+n_lags] / acf[len(returns_eth)-1]
ax3.bar(range(n_lags+1), acf[:n_lags+1], alpha=0.7)
ax3.axhline(y=0, color='k', linestyle='-')
ax3.axhline(y=1.96/np.sqrt(len(returns_eth)), color='r', linestyle='--')
ax3.axhline(y=-1.96/np.sqrt(len(returns_eth)), color='r', linestyle='--')
ax3.set_xlabel('Lag')
ax3.set_ylabel('ACF')
ax3.set_title('ETH Autocorrelation')

# Parameter summary
ax4 = axes[1, 1]
ax4.axis('off')
summary_text = f"""
ETH Calibration Results
======================
μ (drift):     {result_eth.params.mu:.6f}
σ (volatility): {result_eth.params.sigma:.6f}
λ (intensity): {result_eth.params.lambda_:.6f}
η (jump rate): {result_eth.params.eta:.6f}
δ (shift):     {result_eth.params.delta:.6f}

Log-likelihood: {result_eth.log_likelihood:.2f}
AIC: {result_eth.aic:.2f}
BIC: {result_eth.bic:.2f}
Converged: {result_eth.converged}
"""
ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace')

plt.suptitle('ETH Calibration Diagnostics', fontsize=14)
plt.tight_layout()
plt.show()

---
# Section 6: Wiener-Hopf Factorization and Scale Functions

## Wiener-Hopf Factorization

For spectrally negative Lévy processes, the **Wiener-Hopf factorization** takes a particularly simple form. The key object is the **right-inverse function** $\Phi(q)$.

### The Root Function $\Phi(q)$

$\Phi(q)$ is uniquely defined as the positive solution to:

$$\psi(\Phi(q)) = q$$

**Properties:**
- $\Phi(0) = 0$
- $\Phi(q)$ is strictly increasing in $q$
- $\Phi(q) \sim \sqrt{2q/\sigma^2}$ as $q \to \infty$ (when $\sigma > 0$)

We compute $\Phi(q)$ numerically using Brent's root-finding method.

In [ ]:
# Create Wiener-Hopf factorization objects using calibrated parameters
wh_eth = WienerHopfFactorization(result_eth.params)
wh_wbtc = WienerHopfFactorization(result_wbtc.params)

# Plot Φ(q)
fig, ax = plt.subplots(figsize=(10, 6))

q_values = np.linspace(0.01, 20, 100)
phi_eth = [wh_eth.phi(q) for q in q_values]
phi_wbtc = [wh_wbtc.phi(q) for q in q_values]

ax.plot(q_values, phi_eth, 'b-', linewidth=2, label='ETH Φ(q)')
ax.plot(q_values, phi_wbtc, 'orange', linewidth=2, label='WBTC Φ(q)')

ax.set_xlabel('q', fontsize=12)
ax.set_ylabel('Φ(q)', fontsize=12)
ax.set_title('Wiener-Hopf Right-Inverse Function Φ(q)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify ψ(Φ(q)) = q
print("Verification: ψ(Φ(q)) = q")
for q in [1.0, 5.0, 10.0]:
    phi_q = wh_eth.phi(q)
    psi_phi = float(np.real(laplace_exponent(phi_q, result_eth.params)))
    print(f"  q={q:.1f}: Φ(q)={phi_q:.4f}, ψ(Φ(q))={psi_phi:.4f}, error={abs(psi_phi-q):.2e}")

## Scale Functions $W^{(q)}(x)$ and $Z^{(q)}(x)$

The **q-scale function** $W^{(q)}(x)$ is fundamental for first-passage problems. It is defined via its Laplace transform:

$$\int_0^\infty e^{-\theta x} W^{(q)}(x) dx = \frac{1}{\psi(\theta) - q}, \quad \theta > \Phi(q)$$

**Properties:**
- $W^{(q)}(x) = 0$ for $x < 0$
- $W^{(q)}$ is strictly increasing on $[0, \infty)$
- $W^{(q)}(0) = 0$ when $\sigma > 0$

The **Z-scale function** is defined as:

$$Z^{(q)}(x) = 1 + q \int_0^x W^{(q)}(y) dy$$

**Properties:**
- $Z^{(q)}(0) = 1$
- $Z^{(q)}$ is increasing

In [ ]:
# Create scale function objects
sf_eth = ScaleFunction(result_eth.params, wh_eth)
sf_wbtc = ScaleFunction(result_wbtc.params, wh_wbtc)

# Plot scale functions for different q values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_values = np.linspace(0.01, 2.0, 100)
q_list = [0.5, 1.0, 2.0, 5.0]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(q_list)))

# W^(q)(x)
ax1 = axes[0]
for q, color in zip(q_list, colors):
    W_values = [sf_eth.W(x, q) for x in x_values]
    ax1.plot(x_values, W_values, color=color, linewidth=2, label=f'q={q}')

ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('W^(q)(x)', fontsize=12)
ax1.set_title('ETH Scale Function W^(q)(x)', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Z^(q)(x)
ax2 = axes[1]
for q, color in zip(q_list, colors):
    Z_values = [sf_eth.Z(x, q) for x in x_values]
    ax2.plot(x_values, Z_values, color=color, linewidth=2, label=f'q={q}')

ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('Z^(q)(x)', fontsize=12)
ax2.set_title('ETH Z-Scale Function Z^(q)(x)', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify properties
print("Verification of scale function properties:")
print(f"  W(0, q=1) = {sf_eth.W(0.0, 1.0):.6f} (should be ≈ 0)")
print(f"  Z(0, q=1) = {sf_eth.Z(0.0, 1.0):.6f} (should be ≈ 1)")

---
# Section 7: First-Hitting Time Theory

## First-Hitting Time Definition

The **first-hitting time** (or first-passage time) to level $b$ is:

$$\tau_b = \inf\{t \geq 0 : X_t \leq b\}$$

In our DeFi context:
- $x = \log(\text{HF}_{\text{current}})$: starting position in log space
- $b = \log(\text{HF}_{\text{liquidation}}) = \log(1) = 0$: liquidation threshold
- $\tau_b$: first time health factor drops to liquidation threshold

**Liquidation occurs** when $\tau_b < \infty$, i.e., the process hits level $b$.

## Laplace Transform of Hitting Time

For spectrally negative Lévy processes starting at $x > b$, the **Laplace transform of the hitting time** has a beautiful closed form:

$$\boxed{\mathbb{E}[e^{-q\tau_b} | X_0 = x] = Z^{(q)}(x-b) - \frac{q}{\Phi(q)} W^{(q)}(x-b)}$$

where:
- $W^{(q)}(\cdot)$: q-scale function
- $Z^{(q)}(\cdot)$: Z-scale function
- $\Phi(q)$: right-inverse of Laplace exponent

This formula is the cornerstone of our liquidation risk analysis.

## Gaver-Stehfest Numerical Laplace Inversion

To obtain $P(\tau_b \leq t)$ from the Laplace transform, we use the **Gaver-Stehfest algorithm**:

$$f(t) \approx \frac{\ln 2}{t} \sum_{k=1}^{N} v_k \cdot F\left(k \cdot \frac{\ln 2}{t}\right)$$

where:
- $F(s)$ is the Laplace transform of $f(t)$
- $v_k$ are the Stehfest weights (precomputed using high-precision arithmetic)
- $N$ is typically 8-12 for good accuracy

The algorithm requires only **real-valued** evaluations of $F(s)$, making it computationally efficient.

In [ ]:
# Demonstrate Laplace inversion accuracy with a known transform
from crypto_fht.core.laplace_inversion import GaverStehfestInverter

# Test: F(s) = 1/(s+a) has inverse f(t) = e^{-at}
a = 2.0
inverter = GaverStehfestInverter(N=10)

def F_exp(s):
    return 1.0 / (s + a)

t_values = np.array([0.5, 1.0, 2.0, 3.0, 5.0])
computed = inverter.invert(F_exp, t_values)
expected = np.exp(-a * t_values)

print("Gaver-Stehfest Inversion Test: F(s) = 1/(s+2) → f(t) = e^{-2t}")
print("-" * 50)
print(f"{'t':>6} {'Computed':>12} {'Expected':>12} {'Rel Error':>12}")
print("-" * 50)
for t, c, e in zip(t_values, computed, expected):
    rel_error = abs(c - e) / e
    print(f"{t:>6.1f} {c:>12.6f} {e:>12.6f} {rel_error:>12.2e}")

In [ ]:
# Create FirstHittingTime objects using calibrated parameters
fht_eth = FirstHittingTime(result_eth.params)
fht_wbtc = FirstHittingTime(result_wbtc.params)

print("FirstHittingTime objects created for ETH and WBTC.")
print(f"\nETH calibrated effective drift: {result_eth.params.effective_drift:.4f}")
print(f"WBTC calibrated effective drift: {result_wbtc.params.effective_drift:.4f}")

In [ ]:
# Compute liquidation probability term structure
t_values = np.linspace(1, TIME_HORIZON_DAYS, TIME_HORIZON_DAYS)

# For initial health factors from our positions
hf_eth = PAIR_1['initial_hf']
hf_wbtc = PAIR_2['initial_hf']

print(f"Computing term structure for HF = {hf_eth:.4f} (USDC/ETH) and HF = {hf_wbtc:.4f} (DAI/WBTC)...")

probs_eth = np.array([fht_eth.from_health_factor(hf_eth, t=t) for t in t_values])
probs_wbtc = np.array([fht_wbtc.from_health_factor(hf_wbtc, t=t) for t in t_values])

print("Done!")

In [ ]:
# Plot term structure comparison
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(t_values, probs_eth, 'b-', linewidth=2, label=f'USDC/ETH (HF={hf_eth:.2f})')
ax.plot(t_values, probs_wbtc, 'orange', linewidth=2, label=f'DAI/WBTC (HF={hf_wbtc:.2f})')

ax.fill_between(t_values, 0, probs_eth, alpha=0.2, color='blue')
ax.fill_between(t_values, 0, probs_wbtc, alpha=0.2, color='orange')

# Mark key time horizons
for t_mark in [30, 60, 90]:
    ax.axvline(x=t_mark, color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Time Horizon (days)', fontsize=12)
ax.set_ylabel('P(Liquidation)', fontsize=12)
ax.set_title('Liquidation Probability Term Structure', fontsize=14)
ax.set_xlim(0, TIME_HORIZON_DAYS)
ax.set_ylim(0, 1)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key values
print("\nLiquidation Probability at Key Time Horizons:")
print("-" * 50)
for t_mark in [30, 60, 90]:
    idx = int(t_mark) - 1
    print(f"  {t_mark} days: USDC/ETH = {probs_eth[idx]:.2%}, DAI/WBTC = {probs_wbtc[idx]:.2%}")

---
# Section 8: Liquidation Risk Analysis for Both Pairs

Now we perform a comprehensive liquidation risk analysis for both position pairs.

In [ ]:
# 90-day liquidation probability at current health factors
prob_90d_eth = fht_eth.from_health_factor(hf_eth, t=90)
prob_90d_wbtc = fht_wbtc.from_health_factor(hf_wbtc, t=90)

print("="*60)
print("90-DAY LIQUIDATION PROBABILITY")
print("="*60)
print(f"\nPair 1: USDC/ETH")
print(f"  Health Factor: {hf_eth:.4f}")
print(f"  90-Day Liquidation Probability: {prob_90d_eth:.2%}")
print(f"\nPair 2: DAI/WBTC")
print(f"  Health Factor: {hf_wbtc:.4f}")
print(f"  90-Day Liquidation Probability: {prob_90d_wbtc:.2%}")

In [ ]:
# Health factor sensitivity analysis
hf_range = np.linspace(1.1, 2.5, 30)

probs_hf_eth = np.array([fht_eth.from_health_factor(hf, t=90) for hf in hf_range])
probs_hf_wbtc = np.array([fht_wbtc.from_health_factor(hf, t=90) for hf in hf_range])

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(hf_range, probs_hf_eth, 'b-', linewidth=2, label='USDC/ETH')
ax.plot(hf_range, probs_hf_wbtc, 'orange', linewidth=2, label='DAI/WBTC')

# Mark current health factors
ax.axvline(x=hf_eth, color='blue', linestyle='--', alpha=0.5)
ax.axvline(x=hf_wbtc, color='orange', linestyle='--', alpha=0.5)

# Mark risk thresholds
for prob_level in [0.05, 0.10, 0.25]:
    ax.axhline(y=prob_level, color='red', linestyle=':', alpha=0.5)
    ax.text(2.45, prob_level + 0.01, f'{prob_level:.0%}', fontsize=9, color='red')

ax.set_xlabel('Health Factor', fontsize=12)
ax.set_ylabel('90-Day Liquidation Probability', fontsize=12)
ax.set_title('Liquidation Probability vs. Initial Health Factor', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Create liquidation heatmaps
hf_values = np.linspace(1.1, 2.5, 20)
time_values = np.linspace(7, 90, 20)

# Compute probability grids
prob_grid_eth = np.zeros((len(hf_values), len(time_values)))
prob_grid_wbtc = np.zeros((len(hf_values), len(time_values)))

for i, hf in enumerate(hf_values):
    for j, t in enumerate(time_values):
        prob_grid_eth[i, j] = fht_eth.from_health_factor(hf, t=t)
        prob_grid_wbtc[i, j] = fht_wbtc.from_health_factor(hf, t=t)

# Plot side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ETH heatmap
ax1 = axes[0]
im1 = ax1.imshow(prob_grid_eth, aspect='auto', origin='lower',
                  extent=[time_values[0], time_values[-1], hf_values[0], hf_values[-1]],
                  cmap='RdYlGn_r', vmin=0, vmax=1)
cbar1 = plt.colorbar(im1, ax=ax1)
cbar1.set_label('P(Liquidation)')
ax1.set_xlabel('Time Horizon (days)')
ax1.set_ylabel('Health Factor')
ax1.set_title('USDC/ETH Liquidation Probability')
ax1.contour(time_values, hf_values, prob_grid_eth, levels=[0.05, 0.10, 0.25, 0.50],
            colors='white', linewidths=0.5)

# WBTC heatmap
ax2 = axes[1]
im2 = ax2.imshow(prob_grid_wbtc, aspect='auto', origin='lower',
                  extent=[time_values[0], time_values[-1], hf_values[0], hf_values[-1]],
                  cmap='RdYlGn_r', vmin=0, vmax=1)
cbar2 = plt.colorbar(im2, ax=ax2)
cbar2.set_label('P(Liquidation)')
ax2.set_xlabel('Time Horizon (days)')
ax2.set_ylabel('Health Factor')
ax2.set_title('DAI/WBTC Liquidation Probability')
ax2.contour(time_values, hf_values, prob_grid_wbtc, levels=[0.05, 0.10, 0.25, 0.50],
            colors='white', linewidths=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Monte Carlo validation
N_PATHS = 10000
T_MC = 90  # days
dt_mc = 1  # daily steps

def monte_carlo_liquidation_prob(params, x0, threshold, T, dt, n_paths, rng):
    """Estimate liquidation probability via Monte Carlo simulation."""
    n_steps = int(T / dt)
    hit_count = 0
    
    for _ in range(n_paths):
        x = x0
        hit = False
        for _ in range(n_steps):
            # Diffusion
            dW = rng.standard_normal() * np.sqrt(dt / 365)
            x += params.mu * (dt / 365) + params.sigma * dW
            
            # Jumps
            n_jumps = rng.poisson(params.lambda_ * (dt / 365))
            if n_jumps > 0:
                jump_sizes = params.delta + rng.exponential(1.0 / params.eta, size=n_jumps)
                x -= np.sum(jump_sizes)
            
            if x <= threshold:
                hit = True
                break
        
        if hit:
            hit_count += 1
    
    return hit_count / n_paths

print(f"Running Monte Carlo simulation ({N_PATHS:,} paths)...")

mc_rng = np.random.default_rng(123)

# ETH
x0_eth = np.log(hf_eth)
mc_prob_eth = monte_carlo_liquidation_prob(
    result_eth.params, x0_eth, 0.0, T_MC, dt_mc, N_PATHS, mc_rng
)

# WBTC
x0_wbtc = np.log(hf_wbtc)
mc_prob_wbtc = monte_carlo_liquidation_prob(
    result_wbtc.params, x0_wbtc, 0.0, T_MC, dt_mc, N_PATHS, mc_rng
)

print("\nMonte Carlo vs. Analytical Comparison (90-day horizon):")
print("-" * 60)
print(f"USDC/ETH:")
print(f"  Analytical: {prob_90d_eth:.4f}")
print(f"  Monte Carlo: {mc_prob_eth:.4f}")
print(f"  Difference: {abs(prob_90d_eth - mc_prob_eth):.4f}")
print(f"\nDAI/WBTC:")
print(f"  Analytical: {prob_90d_wbtc:.4f}")
print(f"  Monte Carlo: {mc_prob_wbtc:.4f}")
print(f"  Difference: {abs(prob_90d_wbtc - mc_prob_wbtc):.4f}")

In [ ]:
# Visualize sample paths with liquidation threshold
def simulate_paths_for_plot(params, x0, T, dt, n_paths, rng):
    """Simulate paths for visualization."""
    n_steps = int(T / dt)
    times = np.linspace(0, T, n_steps + 1)
    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = x0
    
    for i in range(n_paths):
        for j in range(n_steps):
            # Diffusion
            dW = rng.standard_normal() * np.sqrt(dt / 365)
            paths[i, j+1] = paths[i, j] + params.mu * (dt / 365) + params.sigma * dW
            
            # Jumps
            n_jumps = rng.poisson(params.lambda_ * (dt / 365))
            if n_jumps > 0:
                jump_sizes = params.delta + rng.exponential(1.0 / params.eta, size=n_jumps)
                paths[i, j+1] -= np.sum(jump_sizes)
    
    return times, paths

# Simulate paths
plot_rng = np.random.default_rng(456)
times_eth, paths_eth = simulate_paths_for_plot(result_eth.params, x0_eth, 90, 1, 50, plot_rng)
times_wbtc, paths_wbtc = simulate_paths_for_plot(result_wbtc.params, x0_wbtc, 90, 1, 50, plot_rng)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ETH paths
ax1 = axes[0]
for i in range(paths_eth.shape[0]):
    hit_idx = np.where(paths_eth[i] <= 0)[0]
    if len(hit_idx) > 0:
        ax1.plot(times_eth[:hit_idx[0]+1], paths_eth[i, :hit_idx[0]+1], 'r-', alpha=0.3)
    else:
        ax1.plot(times_eth, paths_eth[i], 'b-', alpha=0.3)

ax1.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Liquidation Threshold')
ax1.set_xlabel('Time (days)')
ax1.set_ylabel('Log Health Factor')
ax1.set_title('USDC/ETH Sample Paths')
ax1.legend()
ax1.grid(True, alpha=0.3)

# WBTC paths
ax2 = axes[1]
for i in range(paths_wbtc.shape[0]):
    hit_idx = np.where(paths_wbtc[i] <= 0)[0]
    if len(hit_idx) > 0:
        ax2.plot(times_wbtc[:hit_idx[0]+1], paths_wbtc[i, :hit_idx[0]+1], 'r-', alpha=0.3)
    else:
        ax2.plot(times_wbtc, paths_wbtc[i], 'orange', alpha=0.3)

ax2.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Liquidation Threshold')
ax2.set_xlabel('Time (days)')
ax2.set_ylabel('Log Health Factor')
ax2.set_title('DAI/WBTC Sample Paths')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Blue/Orange paths: Survived 90 days")
print("Red paths: Hit liquidation threshold")

In [ ]:
# Comprehensive risk metrics table
risk_metrics = {
    'Metric': [
        'Initial Health Factor',
        'Log Health Factor',
        'Effective Drift (μ_eff)',
        'Volatility (σ)',
        'Jump Intensity (λ)',
        'Mean Jump Size',
        '30-Day Liquidation Prob',
        '60-Day Liquidation Prob',
        '90-Day Liquidation Prob',
        '90-Day MC Estimate',
    ],
    'USDC/ETH': [
        f"{hf_eth:.4f}",
        f"{x0_eth:.4f}",
        f"{result_eth.params.effective_drift:.4f}",
        f"{result_eth.params.sigma:.4f}",
        f"{result_eth.params.lambda_:.2f}",
        f"{result_eth.params.mean_jump_size:.4f}",
        f"{fht_eth.from_health_factor(hf_eth, 30):.2%}",
        f"{fht_eth.from_health_factor(hf_eth, 60):.2%}",
        f"{prob_90d_eth:.2%}",
        f"{mc_prob_eth:.2%}",
    ],
    'DAI/WBTC': [
        f"{hf_wbtc:.4f}",
        f"{x0_wbtc:.4f}",
        f"{result_wbtc.params.effective_drift:.4f}",
        f"{result_wbtc.params.sigma:.4f}",
        f"{result_wbtc.params.lambda_:.2f}",
        f"{result_wbtc.params.mean_jump_size:.4f}",
        f"{fht_wbtc.from_health_factor(hf_wbtc, 30):.2%}",
        f"{fht_wbtc.from_health_factor(hf_wbtc, 60):.2%}",
        f"{prob_90d_wbtc:.2%}",
        f"{mc_prob_wbtc:.2%}",
    ],
}

risk_df = pd.DataFrame(risk_metrics)
risk_df.set_index('Metric', inplace=True)
risk_df

---
# Section 9: Sensitivity Analysis

Understanding how model parameters affect liquidation risk is crucial for risk management.

In [ ]:
# Jump intensity sensitivity
lambda_range = np.linspace(2, 30, 15)
base_params = result_eth.params

probs_lambda = []
for lam in lambda_range:
    params_mod = LevyParameters(
        mu=base_params.mu,
        sigma=base_params.sigma,
        lambda_=lam,
        eta=base_params.eta,
        delta=base_params.delta,
    )
    fht_mod = FirstHittingTime(params_mod)
    probs_lambda.append(fht_mod.from_health_factor(hf_eth, t=90))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lambda_range, probs_lambda, 'b-o', linewidth=2, markersize=6)
ax.axvline(x=base_params.lambda_, color='red', linestyle='--', label=f'Calibrated λ={base_params.lambda_:.1f}')
ax.set_xlabel('Jump Intensity (λ)', fontsize=12)
ax.set_ylabel('90-Day Liquidation Probability', fontsize=12)
ax.set_title('Sensitivity to Jump Intensity', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Volatility sensitivity
sigma_range = np.linspace(0.3, 1.5, 15)

probs_sigma = []
for sig in sigma_range:
    params_mod = LevyParameters(
        mu=base_params.mu,
        sigma=sig,
        lambda_=base_params.lambda_,
        eta=base_params.eta,
        delta=base_params.delta,
    )
    fht_mod = FirstHittingTime(params_mod)
    probs_sigma.append(fht_mod.from_health_factor(hf_eth, t=90))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sigma_range, probs_sigma, 'g-o', linewidth=2, markersize=6)
ax.axvline(x=base_params.sigma, color='red', linestyle='--', label=f'Calibrated σ={base_params.sigma:.2f}')
ax.set_xlabel('Volatility (σ)', fontsize=12)
ax.set_ylabel('90-Day Liquidation Probability', fontsize=12)
ax.set_title('Sensitivity to Volatility', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Required HF buffer analysis
# What health factor is needed for target liquidation probabilities?

target_probs = [0.01, 0.05, 0.10]
hf_search = np.linspace(1.1, 3.0, 100)

print("Required Health Factor for Target Liquidation Probability (90-day horizon):")
print("="*70)

for target in target_probs:
    # Find HF where liquidation prob crosses target
    probs_search_eth = [fht_eth.from_health_factor(hf, t=90) for hf in hf_search]
    probs_search_wbtc = [fht_wbtc.from_health_factor(hf, t=90) for hf in hf_search]
    
    # Find crossing point
    idx_eth = np.searchsorted(-np.array(probs_search_eth), -target)
    idx_wbtc = np.searchsorted(-np.array(probs_search_wbtc), -target)
    
    hf_req_eth = hf_search[min(idx_eth, len(hf_search)-1)]
    hf_req_wbtc = hf_search[min(idx_wbtc, len(hf_search)-1)]
    
    print(f"\nTarget: <{target:.0%} liquidation probability")
    print(f"  USDC/ETH: HF ≥ {hf_req_eth:.2f}")
    print(f"  DAI/WBTC: HF ≥ {hf_req_wbtc:.2f}")

In [ ]:
# Comprehensive sensitivity grid
sigma_vals = [0.5, 0.8, 1.0]
lambda_vals = [6, 12, 18]

print("90-Day Liquidation Probability Grid (HF=1.33):")
print("="*50)
print(f"{'σ \\ λ':>10}", end='')
for lam in lambda_vals:
    print(f"{lam:>12}", end='')
print()
print("-"*50)

for sig in sigma_vals:
    print(f"{sig:>10.1f}", end='')
    for lam in lambda_vals:
        params_mod = LevyParameters(
            mu=base_params.mu,
            sigma=sig,
            lambda_=lam,
            eta=base_params.eta,
            delta=base_params.delta,
        )
        fht_mod = FirstHittingTime(params_mod)
        prob = fht_mod.from_health_factor(hf_eth, t=90)
        print(f"{prob:>12.2%}", end='')
    print()

---
# Section 10: Conclusions and Recommendations

## Risk Comparison Summary

In [ ]:
# Final comparison
print("="*70)
print("FINAL RISK COMPARISON")
print("="*70)
print("\nStrategy: Long Stablecoin Collateral / Short Volatile Asset Debt")
print("\nPosition Details:")
print(f"  USDC/ETH: $100k USDC collateral, $60k ETH debt")
print(f"  DAI/WBTC: $100k DAI collateral, $60k WBTC debt")
print("\n90-Day Liquidation Risk:")
print(f"  USDC/ETH: {prob_90d_eth:.1%} (HF={hf_eth:.2f})")
print(f"  DAI/WBTC: {prob_90d_wbtc:.1%} (HF={hf_wbtc:.2f})")

if prob_90d_eth < prob_90d_wbtc:
    safer = "USDC/ETH"
    riskier = "DAI/WBTC"
else:
    safer = "DAI/WBTC"
    riskier = "USDC/ETH"

print(f"\nConclusion: {safer} is the safer position at current leverage.")

## Health Factor Targets by Risk Tolerance

Based on our analysis, we recommend the following minimum health factors for different risk tolerances over a 90-day horizon:

| Risk Tolerance | Max 90-Day Liq. Prob. | USDC/ETH Min HF | DAI/WBTC Min HF |
|----------------|----------------------|-----------------|------------------|
| Conservative   | <1%                  | ~2.5            | ~2.5             |
| Moderate       | <5%                  | ~2.0            | ~2.0             |
| Aggressive     | <10%                 | ~1.7            | ~1.7             |

**Note**: These are illustrative values based on calibrated parameters. Actual values will vary with market conditions.

## Limitations and Extensions

### Model Limitations

1. **Constant parameters**: We assume Lévy parameters are time-invariant. In reality, volatility and jump intensity vary with market conditions.

2. **No correlation modeling**: Collateral and debt assets may have correlated price movements, especially during market stress (wrong-way risk).

3. **Discrete monitoring**: The continuous-time FHT formula assumes continuous monitoring, while Aave liquidations are discrete.

4. **Parameter uncertainty**: Calibrated parameters have estimation error that propagates to risk estimates.

### Potential Extensions

1. **Time-varying parameters**: Regime-switching models or stochastic volatility

2. **Multi-asset correlation**: Joint Lévy processes with shared jump components

3. **Discrete barrier monitoring**: Corrections for discrete observation times

4. **Bayesian calibration**: Quantify parameter uncertainty and propagate to risk bounds

## References

1. Kyprianou, A.E. (2014). *Fluctuations of Lévy Processes with Applications*. Springer.

2. Kuznetsov, A., Kyprianou, A.E., & Rivero, V. (2012). "The Theory of Scale Functions for Spectrally Negative Lévy Processes." *Lévy Matters II*, Springer.

3. Stehfest, H. (1970). "Algorithm 368: Numerical Inversion of Laplace Transforms." *Communications of the ACM*, 13(1), 47-49.

4. Aave Protocol Documentation: https://docs.aave.com/